# Módulo 3: Transformação, Qualidade e Modelagem Dimensional (Exercícios)

Seis exercícios curtos e práticos, cobrindo todas as etapas apresentadas no módulo:

1. **ETL em memória com Pandas: Data Wrangling** (exercício 1)
2. **Validações e testes automáticos de qualidade** (exercício 2)
3. **ELT dentro do DuckDB** (exercício 3)
4. **OLAP: CTE, Window Function e consulta direta a Parquet** (exercício 4)
5. **Modelagem Dimensional (Star Schema)** (exercício 5)
6. **Criação de Data Mart** (exercício 6)

Cada exercício tem uma célula para a sua resposta e, logo depois, uma célula **"Confira sua resposta"**. Se ela imprimir `✅`, a resposta está certa; se der erro, revise o código.

Todos os testes usam a base simplificada `exercicios/modulo03/vendas_brutas_ex.csv` e um banco DuckDB **em memória**, sem alterar o `data_warehouse.duckdb` do módulo principal.

In [ ]:
# Se estiver em um ambiente novo, instale as dependências descomentando a linha abaixo:
# %pip install pandas duckdb pyarrow

from pathlib import Path
import pandas as pd
import duckdb

con = duckdb.connect(database=":memory:")
df_bruto = pd.read_csv("exercicios/modulo03/vendas_brutas_ex.csv")

print(f"Registros brutos carregados: {len(df_bruto)}")
display(df_bruto)

## Parte 1: ETL em memória (Pandas) e Qualidade de Dados

### Exercício 1: Data Wrangling (limpeza, enriquecimento e Parquet)

A partir de `df_bruto`, produza o DataFrame `df_limpo` aplicando as seguintes transformações em Pandas:

1. Remova espaços extras no início e fim dos textos em `cliente` e `produto` (`.str.strip()`);
2. Padronize `categoria` para que existam apenas `'Eletrônicos'` e `'Acessórios'` (corrigindo `'eletronicos'` e `'acessorios'`);
3. Remova vendas duplicadas considerando `id_venda` (`keep="first"`);
4. Converta `data_venda` para `datetime` (`pd.to_datetime`);
5. Crie a coluna `valor_total` calculada como `(quantidade * preco_unitario) - desconto`;
6. Salve `df_limpo` em `exercicios/modulo03/vendas_limpas_ex.parquet` (`engine="pyarrow"`, `index=False`).

*Dica:* você pode usar `.str.strip().replace({"eletronicos": "Eletrônicos", "acessorios": "Acessórios"})`.

In [ ]:
df_limpo = df_bruto.copy()

# escreva suas transformações em df_limpo aqui

# salve em exercicios/modulo03/vendas_limpas_ex.parquet

display(df_limpo)

In [ ]:
# Confira sua resposta (não altere esta célula)
assert len(df_limpo) == 10, f"Após remover duplicados deveriam restar 10 linhas, obtido {len(df_limpo)}"
assert set(df_limpo["categoria"].unique()) == {"Eletrônicos", "Acessórios"}, \
    f"Categorias esperadas: Eletrônicos e Acessórios. Obtido: {set(df_limpo['categoria'].unique())}"
assert df_limpo["cliente"].str.strip().equals(df_limpo["cliente"]), "Ainda há espaços extras em 'cliente'"
assert pd.api.types.is_datetime64_any_dtype(df_limpo["data_venda"]), "data_venda deve estar em formato datetime"
assert "valor_total" in df_limpo.columns and df_limpo["valor_total"].sum() == 3605, \
    "A soma de valor_total deveria ser 3605"
assert Path("exercicios/modulo03/vendas_limpas_ex.parquet").exists(), \
    "Faltou salvar exercicios/modulo03/vendas_limpas_ex.parquet"
print("✅ Exercício 1 correto")

### Exercício 2: testes automáticos de qualidade

Escreva a função `checar_qualidade(df)` que recebe um DataFrame de vendas e devolve um dicionário com 4 verificações booleanas (`True` se passou, `False` se falhou):

- `"ids_unicos"`: não existem `id_venda` duplicados;
- `"sem_nulos"`: não há valores nulos nas colunas `["id_venda", "cliente", "produto", "valor_total"]`;
- `"quantidades_positivas"`: todas as linhas têm `quantidade > 0`;
- `"desconto_valido"`: em todas as linhas, `desconto >= 0` **e** `desconto <= quantidade * preco_unitario`.

*Dicas:* `not df["id_venda"].duplicated().any()` e `(df["quantidade"] > 0).all()`.

In [ ]:
def checar_qualidade(df):
    resultados = {}
    # preencha as 4 regras aqui
    return resultados


resultado_ok = checar_qualidade(df_limpo)
for regra, passou in resultado_ok.items():
    print(f"{regra}: {'APROVADO' if passou else 'REPROVADO'}")

In [ ]:
# Confira sua resposta (não altere esta célula)
res_limpo = checar_qualidade(df_limpo)
assert set(res_limpo.keys()) == {"ids_unicos", "sem_nulos", "quantidades_positivas", "desconto_valido"}, \
    "O dicionário deve ter exatamente as 4 chaves pedidas"
assert all(res_limpo.values()), f"df_limpo deveria passar em tudo: {res_limpo}"

# Teste de falha 1: ID duplicado
df_dup = pd.concat([df_limpo, df_limpo.iloc[[0]]], ignore_index=True)
assert checar_qualidade(df_dup)["ids_unicos"] is False, "Deveria reprovar ID duplicado"

# Teste de falha 2: quantidade zerada
df_qtd = df_limpo.copy()
df_qtd.loc[0, "quantidade"] = 0
assert checar_qualidade(df_qtd)["quantidades_positivas"] is False, "Deveria reprovar quantidade <= 0"

# Teste de falha 3: desconto inválido (negativo ou maior que o valor bruto)
df_desc = df_limpo.copy()
df_desc.loc[0, "desconto"] = -5
assert checar_qualidade(df_desc)["desconto_valido"] is False, "Deveria reprovar desconto negativo"
df_desc.loc[0, "desconto"] = 99_999
assert checar_qualidade(df_desc)["desconto_valido"] is False, "Deveria reprovar desconto maior que o bruto"

print("✅ Exercício 2 correto")

## Parte 2: ELT e SQL Analítico (OLAP) no DuckDB

### Exercício 3: transformação ELT dentro do DuckDB

No padrão **ELT**, primeiro carregamos o CSV bruto para a tabela `vendas_raw_ex` (já feito no código inicial abaixo). Agora escreva o SQL para criar a tabela `vendas_tratadas_ex` diretamente dentro do DuckDB:

- `id_venda`
- `CAST(data_venda AS DATE) AS data_venda`
- `id_cliente`
- `TRIM(cliente) AS cliente` e `TRIM(produto) AS produto`
- `categoria` padronizada para `'Eletrônicos'` e `'Acessórios'` com `CASE`
- `quantidade`, `preco_unitario`, `desconto`
- `(quantidade * preco_unitario) - desconto AS valor_total`
- Remova duplicados por `id_venda` usando `QUALIFY ROW_NUMBER() OVER (PARTITION BY id_venda ORDER BY id_venda) = 1`.

In [ ]:
# Load: carrega o CSV bruto para dentro do DuckDB
con.execute("""
    CREATE OR REPLACE TABLE vendas_raw_ex AS
    SELECT * FROM read_csv_auto('exercicios/modulo03/vendas_brutas_ex.csv')
""")

sql_ex3 = """
-- CREATE OR REPLACE TABLE vendas_tratadas_ex AS ...
"""

con.execute(sql_ex3)
display(con.execute("SELECT * FROM vendas_tratadas_ex ORDER BY id_venda").df())

In [ ]:
# Confira sua resposta (não altere esta célula)
df_ex3 = con.execute("SELECT * FROM vendas_tratadas_ex").df()
assert len(df_ex3) == 10, f"vendas_tratadas_ex deveria ter 10 linhas, obtido {len(df_ex3)}"
assert set(df_ex3["categoria"].unique()) == {"Eletrônicos", "Acessórios"}, \
    "As categorias devem ser apenas 'Eletrônicos' e 'Acessórios'"
assert df_ex3["cliente"].str.strip().equals(df_ex3["cliente"]), "Remova espaços extras de 'cliente' com TRIM"
assert df_ex3["valor_total"].sum() == 3605, "A soma de valor_total deveria ser 3605"
print("✅ Exercício 3 correto")

### Exercício 4: CTE, Window Function e consulta direta a Parquet

Usando **diretamente** o arquivo Parquet criado no Exercício 1 (`read_parquet('exercicios/modulo03/vendas_limpas_ex.parquet')`), escreva uma consulta SQL que:

1. Use uma **CTE** (`WITH ... AS (...)`) para ranquear as vendas dentro de cada categoria da maior para a menor `valor_total` usando `ROW_NUMBER() OVER (PARTITION BY categoria ORDER BY valor_total DESC, id_venda ASC) AS ranking_categoria`;
2. Na consulta principal, filtre apenas as **2 maiores vendas de cada categoria** (`WHERE ranking_categoria <= 2`);
3. Retorne as colunas `id_venda`, `produto`, `categoria`, `valor_total`, `ranking_categoria`, ordenadas por `categoria, ranking_categoria`.

In [ ]:
sql_ex4 = """
-- WITH vendas_ranqueadas AS ( ... )
-- SELECT ... FROM vendas_ranqueadas WHERE ranking_categoria <= 2 ...
"""

df_ex4 = con.execute(sql_ex4).df()
display(df_ex4)

In [ ]:
# Confira sua resposta (não altere esta célula)
colunas_ex4 = {"id_venda", "produto", "categoria", "valor_total", "ranking_categoria"}
assert colunas_ex4.issubset(df_ex4.columns), f"A consulta precisa retornar {colunas_ex4}"
assert len(df_ex4) == 4, f"Top 2 de 2 categorias deveria retornar 4 linhas, obtido {len(df_ex4)}"
assert set(df_ex4["ranking_categoria"]) == {1, 2}, "ranking_categoria deve conter apenas 1 e 2"

elet = df_ex4[df_ex4["categoria"] == "Eletrônicos"].set_index("ranking_categoria")
aces = df_ex4[df_ex4["categoria"] == "Acessórios"].set_index("ranking_categoria")
assert elet.loc[1, "id_venda"] == 8 and elet.loc[1, "valor_total"] == 1100, "Top 1 de Eletrônicos é id_venda=8 (1100)"
assert elet.loc[2, "id_venda"] == 9 and elet.loc[2, "valor_total"] == 900, "Top 2 de Eletrônicos é id_venda=9 (900)"
assert aces.loc[1, "id_venda"] == 5 and aces.loc[1, "valor_total"] == 300, "Top 1 de Acessórios é id_venda=5 (300)"
assert aces.loc[2, "id_venda"] == 2 and aces.loc[2, "valor_total"] == 90, "Top 2 de Acessórios é id_venda=2 (90)"
print("✅ Exercício 4 correto")

## Parte 3: Modelagem Dimensional (Star Schema) e Data Marts

Para agilizar, a célula abaixo já cria as dimensões `dim_cliente` e `dim_data` a partir de `vendas_tratadas_ex`. Nos dois últimos exercícios, você vai completar o Star Schema e construir um Data Mart.

In [ ]:
# Dimensões auxiliares já criadas para você usar como referência
con.execute("""
    CREATE OR REPLACE TABLE dim_cliente AS
    SELECT DISTINCT
        id_cliente,
        cliente AS nome_cliente
    FROM vendas_tratadas_ex;
""")

con.execute("""
    CREATE OR REPLACE TABLE dim_data AS
    SELECT DISTINCT
        CAST(STRFTIME(data_venda, '%Y%m%d') AS INTEGER) AS id_data,
        data_venda,
        EXTRACT(DAY FROM data_venda) AS dia,
        EXTRACT(MONTH FROM data_venda) AS mes,
        EXTRACT(YEAR FROM data_venda) AS ano
    FROM vendas_tratadas_ex;
""")

display(con.execute("SELECT * FROM dim_cliente ORDER BY id_cliente").df())
display(con.execute("SELECT * FROM dim_data ORDER BY data_venda").df())

### Exercício 5: completando o Star Schema (`dim_produto` e `fato_vendas`)

Crie as duas tabelas restantes do modelo estrela a partir de `vendas_tratadas_ex`:

1. **`dim_produto`**: colunas `id_produto` (gerada com `ROW_NUMBER() OVER (ORDER BY produto, categoria)` sobre os pares distintos de `produto` e `categoria`), `nome_produto` e `categoria`;
2. **`fato_vendas`**: colunas `id_venda`, `id_cliente`, `id_produto`, `id_data`, `quantidade`, `preco_unitario`, `desconto`, `valor_total`, relacionando `vendas_tratadas_ex` com `dim_produto`, `dim_cliente` e `dim_data`.

*Dica:* para ligar `vendas_tratadas_ex` com `dim_data`, compare `CAST(STRFTIME(v.data_venda, '%Y%m%d') AS INTEGER) = d.id_data`.

In [ ]:
sql_dim_produto = """
-- CREATE OR REPLACE TABLE dim_produto AS ...
"""

sql_fato_vendas = """
-- CREATE OR REPLACE TABLE fato_vendas AS ...
"""

con.execute(sql_dim_produto)
con.execute(sql_fato_vendas)

display(con.execute("SELECT * FROM dim_produto ORDER BY id_produto").df())
display(con.execute("SELECT * FROM fato_vendas ORDER BY id_venda").df())

In [ ]:
# Confira sua resposta (não altere esta célula)
df_prod = con.execute("SELECT * FROM dim_produto").df()
df_fato = con.execute("SELECT * FROM fato_vendas").df()

assert {"id_produto", "nome_produto", "categoria"}.issubset(df_prod.columns), \
    "dim_produto precisa ter id_produto, nome_produto e categoria"
assert len(df_prod) == 7, f"Deveriam existir 7 produtos distintos em dim_produto, obtido {len(df_prod)}"

colunas_fato = {"id_venda", "id_cliente", "id_produto", "id_data", "quantidade", "preco_unitario", "desconto", "valor_total"}
assert colunas_fato.issubset(df_fato.columns), f"fato_vendas precisa ter as colunas {colunas_fato}"
assert len(df_fato) == 10, f"fato_vendas deveria ter 10 linhas, obtido {len(df_fato)}"
assert df_fato["valor_total"].sum() == 3605, "A soma de valor_total na fato_vendas deveria ser 3605"

# Confere integridade dos JOINs no Star Schema
df_join = con.execute("""
    SELECT c.nome_cliente, SUM(f.valor_total) AS total
    FROM fato_vendas f
    JOIN dim_cliente c ON f.id_cliente = c.id_cliente
    JOIN dim_produto p ON f.id_produto = p.id_produto
    JOIN dim_data d    ON f.id_data = d.id_data
    GROUP BY c.nome_cliente
""").df().set_index("nome_cliente")
assert df_join.loc["Juliana Alves", "total"] == 1500, "Join do Star Schema não bateu para Juliana Alves"
print("✅ Exercício 5 correto")

### Exercício 6: construindo o Data Mart Mensal

Agora que o Star Schema está pronto, crie um Data Mart chamado `mart_vendas_mensal` juntando `fato_vendas`, `dim_produto` e `dim_data`, com uma linha por mês e categoria:

- `ano`
- `mes`
- `categoria`
- `quantidade_vendida`: soma de `quantidade`
- `faturamento_total`: soma de `valor_total`

Ordene o resultado por `ano, mes, categoria`.

**Para discutir:** por que uma ferramenta de BI ou um analista de negócios ganha velocidade e segurança consultando `mart_vendas_mensal` em vez de consultar diretamente `vendas_raw_ex`?

In [ ]:
sql_ex6 = """
-- CREATE OR REPLACE TABLE mart_vendas_mensal AS ...
"""

con.execute(sql_ex6)
display(con.execute("SELECT * FROM mart_vendas_mensal ORDER BY ano, mes, categoria").df())

In [ ]:
# Confira sua resposta (não altere esta célula)
df_mart = con.execute("SELECT * FROM mart_vendas_mensal ORDER BY ano, mes, categoria").df()
assert {"ano", "mes", "categoria", "quantidade_vendida", "faturamento_total"}.issubset(df_mart.columns), \
    "Faltam colunas em mart_vendas_mensal"
assert len(df_mart) == 4, f"Deveriam existir 4 linhas (2 meses x 2 categorias), obtido {len(df_mart)}"
assert df_mart["faturamento_total"].sum() == 3605, "O faturamento_total do mart deveria somar 3605"

idx = df_mart.set_index(["mes", "categoria"])
assert idx.loc[(5, "Eletrônicos"), "faturamento_total"] == 780, "Maio/Eletrônicos deveria somar 780"
assert idx.loc[(5, "Acessórios"), "faturamento_total"] == 440, "Maio/Acessórios deveria somar 440"
assert idx.loc[(6, "Eletrônicos"), "faturamento_total"] == 2270, "Junho/Eletrônicos deveria somar 2270"
assert idx.loc[(6, "Acessórios"), "faturamento_total"] == 115, "Junho/Acessórios deveria somar 115"
print("✅ Exercício 6 correto")

In [ ]:
con.close()
print("Fim dos exercícios do Módulo 3.")